# directory-pipeline — Surya OCR + Alignment Review

Runs Surya OCR and/or the interactive alignment review tool on Google Colab.

**Before you start:**
- Enable a GPU runtime: Runtime → Change runtime type → T4 GPU (required for Surya)
- If using ngrok as a fallback tunnel, store your token as a Colab secret named `NGROK_TOKEN` (Secrets panel in the left sidebar)

Run the **Setup** cells first, then jump to whichever section you need.

---
## Setup

Run these cells once per session.

In [1]:
# --- Static config — edit once ---
PIPELINE_DIR = "/content/drive/Othercomputers/My_Mac/directory-pipeline"
MODEL = "gemini-2.0-flash"
PORT  = 5000

In [2]:
import ipywidgets as widgets
from IPython.display import display

slug_widget = widgets.Text(
    value="",
    placeholder="e.g. brewers_guide_for_the_united_states_cana_bd047d10",
    description="Volume slug:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="600px"),
)

def _on_change(change):
    global VOLUME
    VOLUME = f"output/{change['new']}"
    print(f"VOLUME set to: {VOLUME}")

slug_widget.observe(_on_change, names="value")
display(slug_widget)

Text(value='', description='Volume slug:', layout=Layout(width='600px'), placeholder='e.g. brewers_guide_for_t…

VOLUME set to: output//Users/joshhadro/github/directory-pipeline/output/lain_healy_s_brooklyn_directory_for_the_1897BPL/1897BPL
VOLUME set to: output/directory-pipeline/output/lain_healy_s_brooklyn_directory_for_the_1897BPL/1897BPL
VOLUME set to: output/lain_healy_s_brooklyn_directory_for_the_1897BPL/1897BPL


In [3]:
from google.colab import drive
import os

drive.mount("/content/drive")
os.chdir(PIPELINE_DIR)
print("Working directory:", os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/Othercomputers/My_Mac/directory-pipeline


In [4]:
# Install dependencies (~2 min on first run — surya-ocr pulls in torch/transformers).
# Colab already has numpy; torch is reinstalled by surya-ocr at the right CUDA version.
%pip install -q google-genai pillow requests python-dotenv flask geopy pyngrok

In [5]:
!pip install -q "surya-ocr==0.17.1" "transformers>=4.56.1,<5.0"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.9/189.9 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 42.4 MB/s eta 0:00:00


---
## Run Surya OCR

Runs Surya bounding-box detection on images in an `output/` volume directory.
Produces `*_surya.json` files used by the alignment step.

Adjust `--batch-size` to fit your GPU memory (T4: 8–16 for regular images, higher for microfilm; ~72 has worked for L4).

In [6]:
#Regular photographic / scan materials
#Set VOLUME_SLUG in the config cell above, then run this.
!time python main.py --batch-size 72 --surya-ocr {VOLUME}


══════════════════════════════════════════════════════════════
  Pipeline: --surya-ocr
  Targets:  1
══════════════════════════════════════════════════════════════

[1/1] lain_healy_s_brooklyn_directory_for_the_1897BPL
  uuid: None  |  slug: lain_healy_s_brooklyn_directory_for_the_1897BPL  |  type: dir
  csv:  output/lain_healy_s_brooklyn_directory_for_the_1897BPL/lain_healy_s_brooklyn_directory_for_the_1897BPL.csv
  imgs: output/lain_healy_s_brooklyn_directory_for_the_1897BPL/

  ── pipeline/run_surya_ocr.py
    $ /usr/bin/python3 /content/drive/Othercomputers/My_Mac/directory-pipeline/pipeline/run_surya_ocr.py output/lain_healy_s_brooklyn_directory_for_the_1897BPL --batch-size 72
Loading Surya models…
2026-05-10 02:46:07.291017: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_O

In [ ]:
# Microfilm / high-contrast materials — larger batches fit in memory
# Uncomment to use instead of the regular cell above.
# !time python main.py --batch-size 64 --surya-ocr {VOLUME}

---
## Run alignment

Aligns Surya bounding boxes with Gemini OCR text. Requires `*_surya.json` and
`*_gemini.json` files to already exist in the volume directory.

In [ ]:
# !time python main.py --align-ocr --model {MODEL} {VOLUME}

---
## Review alignment

Starts the interactive Flask review tool and exposes it to your browser.

**Preferred**: Colab's built-in proxy (no account needed) — run the _Start server_ and _Open via Colab proxy_ cells.

**Fallback**: If the proxy URL doesn't work, use the ngrok cells instead.

In [ ]:
import os
os.path(VOLUME)

NameError: name 'VOLUME' is not defined

In [ ]:
import subprocess

subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True)


subprocess.Popen(
    ["python", "-m", "pipeline.review_alignment",
     VOLUME,
     "--host", "0.0.0.0",
     "--port", str(PORT),
     "--model", MODEL],
    stdout=open("/tmp/review.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f"Server starting on {VOLUME} — run the next cell to watch the log.")

Server starting on output/copyright_ledgers_copyright_records_new_2022657932/ — run the next cell to watch the log.


In [ ]:
# Watch the log. Stop this cell (square button) once you see "Models ready."
!tail -f /tmp/review.log

Error: directory not found: output/go_guide_to_pleasant_motoring_e6743f00
^C


In [ ]:
# Option A: Colab built-in proxy (no account needed — try this first)
from google.colab.output import eval_js
url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
print("Review tool URL:", url)

Review tool URL: https://5000-gpu-t4-s-kkb-use1c2-1wcuccd83gb3o-c.us-east1-2.prod.colab.dev


In [ ]:
# Option B: ngrok tunnel (fallback if the proxy URL doesn't work)
# Requires an NGROK_TOKEN secret set in the Colab Secrets panel (left sidebar).
from pyngrok import ngrok
from google.colab import userdata
ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))

for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)


public_url = ngrok.connect(PORT)
print("Review tool URL:", public_url)

Review tool URL: NgrokTunnel: "https://convulsedly-ascosporic-leonore.ngrok-free.dev" -> "http://localhost:5000"


---
## Troubleshooting

**Server not responding** — check the log:
```python
!cat /tmp/review.log
```

**Port already in use** — kill it and restart the server cell:
```python
!fuser -k 5000/tcp
```

**`ERR_NGROK_324` — too many tunnels** — kill old ones, then re-run the ngrok cell:
```python
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)
```

**Clicking Done in the UI shuts down the server.** To restart, re-run the _Start server_ and _Open via proxy/ngrok_ cells.

In [ ]:
import subprocess
from pathlib import Path

base = Path(PIPELINE_DIR) / "output/green_books_and_related"

for collection in sorted(base.iterdir()):
    if not collection.is_dir() or collection.name.startswith("_"):
        continue
    print(f"\n{'='*60}\n{collection.name}\n{'='*60}")
    result = subprocess.run(
        ["python", "pipeline/run_surya_ocr.py", str(collection), "--batch-size", "128"],
        cwd=PIPELINE_DIR,
    )



go_guide_to_pleasant_motoring_e6743f00


KeyboardInterrupt: 


go_guide_to_pleasant_motoring_e6743f00

hackley_harrison_s_hotel_and_apartment_g_4f7822b0

n_h_a_directory_and_guide_to_travelers_b33397d0

smith_s_tourist_guide_of_necessary_infor_e20bf5b0

the_green_book_9ea5d5b0

the_travelers_guide_e088efa0

travelguide_634f3af0
Server starting on output/go_guide_to_pleasant_motoring_e6743f00 — run the next cell to watch the log.


In [ ]:
import subprocess
from pathlib import Path


result = subprocess.run(
    ["python", "pipeline/run_surya_ocr.py", str(collection), "--batch-size", "4"],
    cwd=PIPELINE_DIR,
    capture_output=True,
    text=True,
)
Path("/tmp/surya_err.txt").write_text(result.stderr)
print("Return code:", result.returncode)
print("STDOUT:", result.stdout)


NameError: name 'collection' is not defined

In [ ]:
!tail -50 /tmp/surya_err.txt

tail: cannot open '/tmp/surya_err.txt' for reading: No such file or directory


All volumes

In [ ]:
import subprocess

subprocess.run([
    "rsync", "-av", "--progress",
    "--include=*/",
    "--include=*_aligned.json",
    "--include=*.jpg",
    "--exclude=*_viz.jpg",
    "--exclude=*",
    "/content/drive/Othercomputers/My_Mac/directory-pipeline/output/green_books_and_related/",
    "/content/review_work/"
])


KeyboardInterrupt: 

In [ ]:
!du -sh /content/review_work/


In [ ]:
import subprocess
from google.colab.output import eval_js

#OUTPUT_DIR = f"{PIPELINE_DIR}/output/green_books_and_related"
OUTPUT_DIR = "/content/review_work"

subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True)

subprocess.Popen(
    ["python", "-m", "pipeline.review_alignment",
     OUTPUT_DIR,
     "--host", "0.0.0.0",
     "--port", str(PORT),
     "--model", MODEL],
    stdout=open("/tmp/review.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print(f"Server starting — run the next cell to watch the log.")
print(f"URL: {eval_js(f'google.colab.kernel.proxyPort({PORT})')}")


Server starting — run the next cell to watch the log.
URL: https://5000-gpu-t4-s-kkb-use1c2-1wcuccd83gb3o-c.us-east1-2.prod.colab.dev


In [ ]:
!cat /tmp/review.log | tail -50

  images: /content/drive/Othercomputers/My_Mac/directory-pipeline/output/green_books_and_related
  model:  gemini-2.0-flash
Loading Surya models… (this takes ~30 s)
2026-03-17 01:43:01.099465: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-17 01:43:01.118344: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773711781.141904    8597 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773711781.149683    8597 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has al

Ngrok version


---
## Review all Green Books volumes (sequential)

Discovers all volumes under `output/green_books_and_related/` and lets you
step through them one at a time. Select a volume, click **▶ Start Review**,
and open the URL shown. When you're done, pick the next volume and click again.

Run the **Setup** cells first if you haven't already.

In [ ]:
import subprocess
import time
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab.output import eval_js

COLLECTION_DIR = Path(PIPELINE_DIR) / "output" / "green_books_and_related"

# Discover volumes (top-level subdirs only)
volumes = sorted(
    d for d in COLLECTION_DIR.iterdir()
    if d.is_dir() and not d.name.startswith("_")
)

if not volumes:
    print(f"No volumes found under {COLLECTION_DIR}")
else:
    print(f"Found {len(volumes)} volume(s) in {COLLECTION_DIR.name}/:")
    for v in volumes:
        n = len(list(v.rglob("*_aligned.json")))
        print(f"  {v.name}  ({n} aligned JSON file(s))")

_reviewed = set()  # track which volumes were started this session

vol_dropdown = widgets.Dropdown(
    options=[(v.name, v) for v in volumes],
    description="Volume:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="720px"),
)

start_btn = widgets.Button(
    description="\u25b6 Start Review",
    button_style="primary",
    layout=widgets.Layout(width="160px"),
)

progress_label = widgets.Label(value=f"Started this session: 0/{len(volumes)}")
status_out = widgets.Output()

def on_start(_b):
    vol = vol_dropdown.value
    with status_out:
        clear_output()
        print(f"Stopping any existing server on port {PORT}\u2026")
        subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True)
        time.sleep(1)

        n_aligned = len(list(vol.rglob("*_aligned.json")))
        print(f"Starting: {vol.name}")
        print(f"  {n_aligned} aligned JSON file(s) in this volume")

        subprocess.Popen(
            ["python", "-m", "pipeline.review_alignment",
             str(vol),
             "--host", "0.0.0.0",
             "--port", str(PORT),
             "--model", MODEL],
            cwd=PIPELINE_DIR,
            stdout=open("/tmp/review.log", "w"),
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

        time.sleep(3)  # give Flask time to start

        try:
            proxy_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
            print(f"\n\u2713 Review URL (Colab proxy):\n  {proxy_url}\n")
        except Exception:
            print("(Colab proxy URL unavailable \u2014 run the ngrok cell below)")

        print("Check server log:  !cat /tmp/review.log | tail -30")

        _reviewed.add(vol.name)
        progress_label.value = f"Started this session: {len(_reviewed)}/{len(volumes)}"

start_btn.on_click(on_start)

display(widgets.VBox([
    vol_dropdown,
    widgets.HBox([start_btn, widgets.Label("   "), progress_label]),
    status_out,
]))


Found 7 volume(s) in green_books_and_related/:
  go_guide_to_pleasant_motoring_e6743f00  (598 aligned JSON file(s))
  hackley_harrison_s_hotel_and_apartment_g_4f7822b0  (54 aligned JSON file(s))
  n_h_a_directory_and_guide_to_travelers_b33397d0  (72 aligned JSON file(s))
  smith_s_tourist_guide_of_necessary_infor_e20bf5b0  (204 aligned JSON file(s))
  the_green_book_9ea5d5b0  (3424 aligned JSON file(s))
  the_travelers_guide_e088efa0  (86 aligned JSON file(s))
  travelguide_634f3af0  (1232 aligned JSON file(s))


In [ ]:
# Optional: ngrok URL (use if Colab proxy doesn't work)
from pyngrok import ngrok
from google.colab import userdata
ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))

for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

public_url = ngrok.connect(PORT)
print("Review tool URL (ngrok):", public_url)


Review tool URL (ngrok): NgrokTunnel: "https://convulsedly-ascosporic-leonore.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
# Watch the server log — stop this cell once you see "Models ready."
!tail -f /tmp/review.log


2026-03-19 02:46:53.703497: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773888413.744151   14541 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773888413.756011   14541 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773888413.784820   14541 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773888413.784863   14541 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773888413.784868   14541 computation_placer.cc:177] computation placer alr

---
## Review sub-volumes within a large collection

For collections like `the_green_book_9ea5d5b0` and `travelguide_634f3af0` that contain
many individual NYPL items (UUID subdirectories), loading the whole collection at once
is too slow. Use this cell instead to step through one sub-volume (UUID) at a time.

In [ ]:
import subprocess
import time
import re
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab.output import eval_js

_UUID_RE = re.compile(
    r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$', re.I
)

COLLECTION_DIR = Path(PIPELINE_DIR) / "output" / "green_books_and_related"

# Find top-level collection dirs that contain UUID sub-volumes
collections_with_subvols = sorted(
    d for d in COLLECTION_DIR.iterdir()
    if d.is_dir()
    and not d.name.startswith("_")
    and any(_UUID_RE.match(s.name) for s in d.iterdir() if s.is_dir())
)

if not collections_with_subvols:
    print(f"No multi-volume collections found under {COLLECTION_DIR}")
else:
    print(f"Collections with UUID sub-volumes:")
    for c in collections_with_subvols:
        n = sum(1 for s in c.iterdir() if s.is_dir() and _UUID_RE.match(s.name))
        print(f"  {c.name}  ({n} sub-volumes)")

# --- Widgets ---
collection_dropdown = widgets.Dropdown(
    options=[(c.name, c) for c in collections_with_subvols],
    description="Collection:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="720px"),
)

subvol_dropdown = widgets.Dropdown(
    description="Sub-volume:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="720px"),
)

def refresh_subvols(change=None):
    col = collection_dropdown.value
    if col is None:
        subvol_dropdown.options = []
        return
    subs = sorted(
        s for s in col.iterdir()
        if s.is_dir() and _UUID_RE.match(s.name)
    )
    subvol_dropdown.options = [
        (f"{s.name}  ({len(list(s.rglob('*_aligned.json')))} aligned)", s)
        for s in subs
    ]

collection_dropdown.observe(refresh_subvols, names="value")
refresh_subvols()  # populate on load

start_btn = widgets.Button(
    description="\u25b6 Start Review",
    button_style="primary",
    layout=widgets.Layout(width="160px"),
)

_reviewed_subs = set()
progress_label = widgets.Label()

def _update_progress():
    total = len(subvol_dropdown.options)
    progress_label.value = f"Started this session: {len(_reviewed_subs)}/{total}"

_update_progress()

status_out = widgets.Output()

def on_start(_b):
    sub = subvol_dropdown.value
    if sub is None:
        with status_out:
            print("No sub-volume selected.")
        return
    with status_out:
        clear_output()
        print(f"Stopping any existing server on port {PORT}\u2026")
        subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True)
        time.sleep(1)

        n_aligned = len(list(sub.rglob("*_aligned.json")))
        print(f"Starting: {sub.name}")
        print(f"  {n_aligned} aligned JSON file(s)")

        subprocess.Popen(
            ["python", "-m", "pipeline.review_alignment",
             str(sub),
             "--host", "0.0.0.0",
             "--port", str(PORT),
             "--model", MODEL],
            cwd=PIPELINE_DIR,
            stdout=open("/tmp/review.log", "w"),
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

        time.sleep(3)

        try:
            proxy_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
            print(f"\n\u2713 Review URL (Colab proxy):\n  {proxy_url}\n")
        except Exception:
            print("(Colab proxy URL unavailable \u2014 run the ngrok cell below)")

        print("Check server log:  !cat /tmp/review.log | tail -30")

        _reviewed_subs.add(sub.name)
        _update_progress()

start_btn.on_click(on_start)

display(widgets.VBox([
    collection_dropdown,
    subvol_dropdown,
    widgets.HBox([start_btn, widgets.Label("   "), progress_label]),
    status_out,
]))


Stopping any existing server on port 5000…
Starting: 6c5ccbe0-1a73-0132-b81d-58d385a7bbd0
  168 aligned JSON file(s)

✓ Review URL (Colab proxy):
  https://5000-gpu-t4-s-kkb-usw1b2-y6cco41g1gpt-b.us-west1-2.prod.colab.dev

Check server log:  !cat /tmp/review.log | tail -30
